In [ ]:
# Importing necessary classes


import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import boxcox
from scipy.special import inv_boxcox
import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM, Dropout
import tensorflow as tf
import seaborn as sn

In [ ]:
# Loading the dataset and shaping it into a time series dataset
from pathlib import Path

csv_candidates = ['content/2619492.csv', '2619492.csv', 'Data/2619492.csv']
csv_path = next((p for p in csv_candidates if Path(p).exists()), None)
if csv_path is None:
    raise FileNotFoundError('Could not find 2619492.csv in expected paths.')

rainfall = pd.read_csv(csv_path)
rainfall["DATE"] = pd.to_datetime(rainfall["DATE"], dayfirst=True)
rainfall = rainfall.set_index('DATE')
rainfall.head(3)

In [ ]:
#Plotting the histogram of dataset
plt.figure(figsize=(11, 6))
plt.rc('font', size=20)         # controls default text sizes
plt.rc('axes', titlesize=22)    # fontsize of the axes title
plt.rc('axes', labelsize=22)    # fontsize of the x and y labels
plt.rc('xtick', labelsize=20)   # fontsize of the tick labels
plt.rc('ytick', labelsize=20)  # fontsize of the tick labels
plt.rc('legend', fontsize=22)   # legend fontsize
plt.rc('figure', titlesize=23)  # fontsize of the figure title
series = rainfall.PRCP
series.hist(bins=50)
plt.show()

In [ ]:
# Box-Cox Transformation Function
def boxcox1p(data):
  p1data= data+1
  bcdata, bclambda = boxcox(p1data)
  return bcdata, bclambda

In [ ]:
# Inverse Box-Cox transformation Function
def inv_boxcox1p(bcdata, bclambda):
  data=inv_boxcox(bcdata, bclambda)-1
  return data

In [ ]:
#Calling the Box-Cox transformation function to normalize the data distribution
inv_data, lam_value = boxcox1p(rainfall.PRCP)
plt.hist(inv_data, bins=100)
plt.show()

In [ ]:
# Use Box-Cox-transformed data directly (no Min-Max scaling)
rain_data = inv_data

In [ ]:
#Function for making windowed dataset
from numpy import array
def split_sequences(series, window):
  x, y= list(), list()
  for i in range(len(series)):
    end= i+ window
    if end>len(series)-1:
      break
    seq_x, seq_y= series[i:end], series[end]
    x.append(seq_x)
    y.append(seq_y)
  return array(x), array(y)

In [ ]:
#Dividing the dataset into train and test subsets
train_data = rain_data[0:9058]
valid_data = rain_data[9058:]
window_size = 70
x_train, y_train = split_sequences(train_data, window_size)
x_test, y_test = split_sequences(valid_data, window_size)

In [ ]:
#Reshaping the test and test input data for LSTM model
x_train = np.reshape(x_train, (x_train.shape[0], x_train.shape[1], 1))
x_test = np.reshape(x_test, (x_test.shape[0], x_test.shape[1], 1))

In [ ]:
# Building the LSTM model
model = Sequential()
model.add(LSTM(32, input_shape=(window_size, 1), return_sequences=True))
model.add(Dropout(0.5))
model.add(LSTM(32,))
model.add(Dropout(0.5))
model.add(Dense(16, activation='relu'))
model.add(Dense(1, activation='relu'))

In [ ]:
lr_schedule = tf.keras.callbacks.LearningRateScheduler(
    lambda epoch: 1e-7 * 10**(epoch / 40))
optimizer = tf.keras.optimizers.Adam(learning_rate=1e-3)
model.compile(loss=tf.keras.losses.Huber(),
              optimizer=optimizer,
              metrics=["mean_squared_error"])
print(model.summary())

In [ ]:
# LSTM model training phase
print("Starting to fit")
history = model.fit(x_train, y_train, epochs=100, batch_size=60, verbose=1,
                    validation_data=(x_train, y_train), callbacks=[lr_schedule])

In [ ]:
model.evaluate(x_test, y_test)

In [ ]:
#Predicting the values for test set
result = model.predict(x_test)
result = inv_boxcox1p(result.flatten(), lam_value)
y_valid = inv_boxcox1p(y_test, lam_value)
import math
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
lstm_result = result.flatten()
score_lstm = math.sqrt(mean_squared_error(y_valid, lstm_result))
mae_lstm   = mean_absolute_error(y_valid, lstm_result)
r2_lstm    = r2_score(y_valid, lstm_result)
print(f'LSTM RMSE : {score_lstm:.4f}')
print(f'LSTM MAE  : {mae_lstm:.4f}')
print(f'LSTM R2   : {r2_lstm:.4f}')

In [ ]:
# summarize history for loss
plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.ylabel('loss')
plt.xlabel('epoch')
plt.legend(['train', 'test'], loc='upper left')
plt.show()

In [ ]:
#Defining average and naive forecast functions
def naiveForecast(x_data):
    # Expected Shape of x_data: (number of samples, number of readings per sample, 1)
    # Shape of forecast: (number of samples, )
    forecast = x_data[:,-1,0]
    return forecast

def avgForecast(x_data):
    # Expected Shape of x_data: (number of samples, number of readings per sample, 1)
    # Shape of forecast: (number of samples, )
    forecast = np.mean(x_data[:,:,0], axis=1)
    return forecast

naive_forecast = naiveForecast(x_test)
avg_forecast = avgForecast(x_test)

In [ ]:
avg_forecast = inv_boxcox1p(avg_forecast, lam_value)
naive_forecast = inv_boxcox1p(naive_forecast, lam_value)

In [ ]:
rmse_avg   = math.sqrt(mean_squared_error(y_valid, avg_forecast))
rmse_naive = math.sqrt(mean_squared_error(y_valid, naive_forecast))
print(f'Average RMSE : {rmse_avg:.4f}')
print(f'Naive RMSE   : {rmse_naive:.4f}')

---
## XGBoost Model
We train XGBoost on the **same 70-lag features** used by the LSTM, operating in the **Box-Cox-transformed space** so the pipeline stays
consistent with the baseline preprocessing (Box-Cox only).


In [ ]:
# Install / import XGBoost (already available in Colab)
import xgboost as xgb
print(f'XGBoost version: {xgb.__version__}')

In [ ]:
# ── XGBoost uses the same windowed data but in 2-D (samples × lag features) ──
# x_train / x_test were reshaped to 3-D for LSTM; we need 2-D for XGBoost.
# We rebuild them directly from the Box-Cox-transformed 1-D series.
x_train_xgb = x_train[:, :, 0]   # shape (n_train, 70)
x_test_xgb  = x_test[:, :, 0]    # shape (n_test,  70)
# y_train / y_test are already 1-D (Box-Cox space)
print(f'XGBoost train shape : {x_train_xgb.shape}')
print(f'XGBoost test  shape : {x_test_xgb.shape}')

In [ ]:
# ── Train & tune XGBoost (time-series holdout) ─────────────────────────────
from sklearn.model_selection import ParameterGrid, ParameterSampler

# Last portion of training is used as chronological validation for tuning
n_val = max(256, int(len(x_train_xgb) * 0.12))
x_tr, x_val = x_train_xgb[:-n_val], x_train_xgb[-n_val:]
y_tr, y_val = y_train[:-n_val], y_train[-n_val:]

# Broader coarse search followed by a narrow local refinement around the best
# candidate. This gives XGBoost a better shot on the small precipitation series
# without touching the LSTM baseline.
coarse_space = {
   'n_estimators': [200, 350, 500, 750, 1000],
    'max_depth': [2, 3, 4, 5, 6],
    'learning_rate': [0.01, 0.02, 0.03, 0.05],
    'subsample': [0.65, 0.75, 0.85, 0.95, 1.0],
    'colsample_bytree': [0.6, 0.7, 0.8, 0.9, 1.0],
    'min_child_weight': [1, 2, 3, 5, 8],
    'gamma': [0.0, 0.05, 0.1, 0.2],
    'reg_alpha': [0.0, 0.01, 0.05, 0.1],
    'reg_lambda': [0.8, 1.0, 1.5, 2.0, 3.0]
}

# Evaluate in original (inches) space for practical RMSE
y_val_orig = inv_boxcox1p(y_val, lam_value)
best_val_rmse = np.inf
best_cfg = None
best_model = None

# Sample a manageable coarse set rather than exhaustively walking the full grid.
all_cfg = list(ParameterSampler(coarse_space, n_iter=45, random_state=42))
for i, cfg in enumerate(all_cfg, start=1):
    model_try = xgb.XGBRegressor(
        objective='reg:squarederror',
        eval_metric='rmse',
        random_state=42,
        n_jobs=-1,
        tree_method='hist',
        early_stopping_rounds=80,
        **cfg,
    )
    model_try.fit(
        x_tr, y_tr,
        eval_set=[(x_val, y_val)],
        verbose=False,
    )

    pred_val_bc = model_try.predict(x_val)
    pred_val_orig = inv_boxcox1p(pred_val_bc, lam_value)
    rmse_val = math.sqrt(mean_squared_error(y_val_orig, pred_val_orig))

    if rmse_val < best_val_rmse:
        best_val_rmse = rmse_val
        best_cfg = cfg
        best_model = model_try

total = len(all_cfg)

# Narrow local refinement around the best coarse configuration.
fine_space = {
    'n_estimators': sorted(set([
        max(100, best_cfg['n_estimators'] - 200),
        best_cfg['n_estimators'],
        best_cfg['n_estimators'] + 200,
    ])),
    'max_depth': sorted(set([
        max(2, best_cfg['max_depth'] - 1),
        best_cfg['max_depth'],
        best_cfg['max_depth'] + 1,
    ])),
    'learning_rate': sorted(set([
        round(max(0.005, best_cfg['learning_rate'] / 1.5), 4),
        best_cfg['learning_rate'],
        round(min(0.08, best_cfg['learning_rate'] * 1.5), 4),
    ])),
    'subsample': sorted(set([
        round(max(0.55, best_cfg['subsample'] - 0.1), 2),
        best_cfg['subsample'],
        round(min(1.0, best_cfg['subsample'] + 0.1), 2),
    ])),
    'colsample_bytree': sorted(set([
        round(max(0.5, best_cfg['colsample_bytree'] - 0.1), 2),
        best_cfg['colsample_bytree'],
        round(min(1.0, best_cfg['colsample_bytree'] + 0.1), 2),
    ])),
    'min_child_weight': sorted(set([
        max(1, best_cfg['min_child_weight'] - 1),
        best_cfg['min_child_weight'],
        best_cfg['min_child_weight'] + 1,
    ])),
    'gamma': sorted(set([
        round(max(0.0, best_cfg['gamma'] - 0.05), 3),
        best_cfg['gamma'],
        round(best_cfg['gamma'] + 0.05, 3),
    ])),
    'reg_alpha': sorted(set([
        round(max(0.0, best_cfg['reg_alpha'] - 0.02), 3),
        best_cfg['reg_alpha'],
        round(best_cfg['reg_alpha'] + 0.02, 3),
    ])),
    'reg_lambda': sorted(set([
        round(max(0.5, best_cfg['reg_lambda'] - 0.5), 3),
        best_cfg['reg_lambda'],
        round(best_cfg['reg_lambda'] + 0.5, 3),
    ])),
}

for cfg in ParameterGrid(fine_space):
    model_try = xgb.XGBRegressor(
        objective='reg:squarederror',
        eval_metric='rmse',
        random_state=42,
        n_jobs=-1,
        tree_method='hist',
        early_stopping_rounds=80,
        **cfg,
    )
    model_try.fit(
        x_tr, y_tr,
        eval_set=[(x_val, y_val)],
        verbose=False,
    )

    pred_val_bc = model_try.predict(x_val)
    pred_val_orig = inv_boxcox1p(pred_val_bc, lam_value)
    rmse_val = math.sqrt(mean_squared_error(y_val_orig, pred_val_orig))

    if rmse_val < best_val_rmse:
        best_val_rmse = rmse_val
        best_cfg = cfg
        best_model = model_try

# Refit a final XGBoost model on the full training data using the best hyperparameters
# found on the validation split. This is the model used downstream for test forecasts.
xgb_model = xgb.XGBRegressor(
    objective='reg:squarederror',
    eval_metric='rmse',
    random_state=42,
    n_jobs=-1,
    tree_method='hist',
    **best_cfg,
)
xgb_model.fit(x_train_xgb, y_train, verbose=False)

print('Selected XGBoost params:', best_cfg)
print(f'Best XGBoost validation RMSE (orig scale): {best_val_rmse:.4f}')
print(f'Validation-tuned model best boosting iteration: {best_model.best_iteration}')
print('Final XGBoost model refit on full training data with optimal hyperparameters.')


In [ ]:
# ── XGBoost predictions in Box-Cox space ───────────────────────────────────
xgb_pred_bc = xgb_model.predict(x_test_xgb)          # shape (n_test,)

# Inverse-transform to original precipitation units
xgb_pred_orig = inv_boxcox1p(xgb_pred_bc, lam_value)

score_xgb = math.sqrt(mean_squared_error(y_valid, xgb_pred_orig))
mae_xgb   = mean_absolute_error(y_valid, xgb_pred_orig)
r2_xgb    = r2_score(y_valid, xgb_pred_orig)
print(f'XGBoost RMSE : {score_xgb:.4f}')
print(f'XGBoost MAE  : {mae_xgb:.4f}')
print(f'XGBoost R2   : {r2_xgb:.4f}')

In [ ]:
# ── Feature importance (top-20 lag features) ───────────────────────────────
feat_imp = xgb_model.feature_importances_
top20_idx = np.argsort(feat_imp)[-20:][::-1]
top20_names = [f'lag_{window_size - i}' for i in top20_idx]

plt.figure(figsize=(9, 5))
plt.barh(top20_names[::-1], feat_imp[top20_idx[::-1]], color='steelblue', alpha=0.8)
plt.title('XGBoost — Top 20 Feature Importances')
plt.xlabel('Importance score')
plt.tight_layout()
plt.show()

---
## Ensemble: LSTM + XGBoost (Simple Averaging + other)
Both models predict in the **original (inches) space**. We average their predictions with equal weights (0.5 / 0.5).


In [ ]:
# ── Ensemble comparison: simple average, inverse-RMSE, grid search, median ──
# Validation slice for tuning ensemble weights
n_ens_val = max(256, int(len(x_train) * 0.08))
x_ens_val = x_train[-n_ens_val:]
y_ens_val_bc = y_train[-n_ens_val:]

# Base-model predictions on the validation slice and the test set
lstm_val_pred_orig = inv_boxcox1p(model.predict(x_ens_val, verbose=0).flatten(), lam_value)
xgb_val_pred_orig  = inv_boxcox1p(xgb_model.predict(x_ens_val[:, :, 0]), lam_value)
y_ens_val_orig     = inv_boxcox1p(y_ens_val_bc, lam_value)

lstm_test_pred_orig = lstm_result
xgb_test_pred_orig   = xgb_pred_orig

# Validation RMSE for each base model
rmse_lstm_val = math.sqrt(mean_squared_error(y_ens_val_orig, lstm_val_pred_orig))
rmse_xgb_val   = math.sqrt(mean_squared_error(y_ens_val_orig, xgb_val_pred_orig))

# 1) Simple average
ensemble_simple = 0.5 * lstm_test_pred_orig + 0.5 * xgb_test_pred_orig

# 2) Inverse-RMSE weighting
inv_w_lstm = 1.0 / rmse_lstm_val
inv_w_xgb = 1.0 / rmse_xgb_val
inv_w_sum = inv_w_lstm + inv_w_xgb
w_lstm_inv = inv_w_lstm / inv_w_sum
w_xgb_inv = inv_w_xgb / inv_w_sum
ensemble_inverse_rmse = w_lstm_inv * lstm_test_pred_orig + w_xgb_inv * xgb_test_pred_orig

# 3) Grid-search weight on the validation slice
weight_grid = np.linspace(0.0, 1.0, 101)
best_grid_w_lstm = 0.5
best_grid_rmse = np.inf

for w_lstm in weight_grid:
    val_mix = w_lstm * lstm_val_pred_orig + (1.0 - w_lstm) * xgb_val_pred_orig
    val_rmse = math.sqrt(mean_squared_error(y_ens_val_orig, val_mix))
    if val_rmse < best_grid_rmse:
        best_grid_rmse = val_rmse
        best_grid_w_lstm = float(w_lstm)

best_grid_w_xgb = 1.0 - best_grid_w_lstm
ensemble_grid_search = best_grid_w_lstm * lstm_test_pred_orig + best_grid_w_xgb * xgb_test_pred_orig

# 4) Median blend
# With only two models, the median equals the average for each time step.
ensemble_median = np.median(
    np.column_stack([lstm_test_pred_orig, xgb_test_pred_orig]),
    axis=1,
)

# 5) Standalone XGBoost as a no-meta-learner fallback candidate.
xgb_only_pred = xgb_test_pred_orig

# Metrics for all ensemble variants
ensemble_methods = {
    'Simple average': ensemble_simple,
    'Inverse RMSE weighting': ensemble_inverse_rmse,
    'Grid-search weight': ensemble_grid_search,
    'Median blend': ensemble_median,
    'XGBoost only': xgb_only_pred,
}

ensemble_scores = {}
for name, pred in ensemble_methods.items():
    ensemble_scores[name] = {
        'RMSE': math.sqrt(mean_squared_error(y_valid, pred)),
        'MAE': mean_absolute_error(y_valid, pred),
        'R2': r2_score(y_valid, pred),
    }

# Select the best strategy on validation without using a meta-learner.
validation_strategy_scores = {
    'Simple average': math.sqrt(mean_squared_error(y_ens_val_orig, 0.5 * lstm_val_pred_orig + 0.5 * xgb_val_pred_orig)),
    'Inverse RMSE weighting': math.sqrt(mean_squared_error(y_ens_val_orig, w_lstm_inv * lstm_val_pred_orig + w_xgb_inv * xgb_val_pred_orig)),
    'Grid-search weight': best_grid_rmse,
    'Median blend': math.sqrt(mean_squared_error(y_ens_val_orig, np.median(np.column_stack([lstm_val_pred_orig, xgb_val_pred_orig]), axis=1))),
    'XGBoost only': rmse_xgb_val,
}
best_strategy = min(validation_strategy_scores, key=validation_strategy_scores.get)
ensemble_pred = ensemble_methods[best_strategy]
score_ensemble = ensemble_scores[best_strategy]['RMSE']
mae_ensemble = ensemble_scores[best_strategy]['MAE']
r2_ensemble = ensemble_scores[best_strategy]['R2']

# Also expose each variant explicitly for later comparison and plots.
score_simple = ensemble_scores['Simple average']['RMSE']
score_inverse_rmse = ensemble_scores['Inverse RMSE weighting']['RMSE']
score_grid = ensemble_scores['Grid-search weight']['RMSE']
score_median = ensemble_scores['Median blend']['RMSE']
score_xgb_only = ensemble_scores['XGBoost only']['RMSE']

print(f'Validation RMSE  - LSTM : {rmse_lstm_val:.4f}')
print(f'Validation RMSE  - XGBoost : {rmse_xgb_val:.4f}')
print(f'Inverse-RMSE weights -> LSTM: {w_lstm_inv:.3f}, XGBoost: {w_xgb_inv:.3f}')
print(f'Grid-search weights  -> LSTM: {best_grid_w_lstm:.2f}, XGBoost: {best_grid_w_xgb:.2f}')
print(f'Best ensemble strategy on validation: {best_strategy}')
for name, metrics in ensemble_scores.items():
    print(f"{name} RMSE : {metrics['RMSE']:.4f} | MAE : {metrics['MAE']:.4f} | R2 : {metrics['R2']:.4f}")


In [ ]:
# ── Final comparison table for ensemble strategies ─────────────────────────
comparison_rows = [
    ('LSTM (baseline)', score_lstm, mae_lstm, r2_lstm),
    ('XGBoost (standalone)', score_xgb, mae_xgb, r2_xgb),
    ('Simple average', score_simple, ensemble_scores['Simple average']['MAE'], ensemble_scores['Simple average']['R2']),
    ('Inverse RMSE weighting', score_inverse_rmse, ensemble_scores['Inverse RMSE weighting']['MAE'], ensemble_scores['Inverse RMSE weighting']['R2']),
    ('Grid-search weight', score_grid, ensemble_scores['Grid-search weight']['MAE'], ensemble_scores['Grid-search weight']['R2']),
    ('Median blend', score_median, ensemble_scores['Median blend']['MAE'], ensemble_scores['Median blend']['R2']),
    ('XGBoost only', score_xgb_only, ensemble_scores['XGBoost only']['MAE'], ensemble_scores['XGBoost only']['R2']),
]

df_summary = pd.DataFrame(comparison_rows, columns=['Model', 'RMSE', 'MAE', 'R2'])
df_summary = df_summary.sort_values('RMSE', ascending=True).reset_index(drop=True)
print(df_summary.to_string(index=False, float_format=lambda x: f'{x:.4f}'))

best_model_name = df_summary.iloc[0]['Model']
best_rmse = df_summary.iloc[0]['RMSE']
print()
print(f'Best method by RMSE: {best_model_name} ({best_rmse:.4f})')
print(f'Grid-search weight used on test: LSTM={best_grid_w_lstm:.2f}, XGBoost={best_grid_w_xgb:.2f}')
print(f'Inverse-RMSE weights used on test: LSTM={w_lstm_inv:.3f}, XGBoost={w_xgb_inv:.3f}')
print(f'Selected best strategy: {best_strategy}')


In [ ]:
# ── Forecast comparison plot ───────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(15, 5))
ax.plot(y_valid, label='Actual', color='black', linewidth=1.2, alpha=0.8)
ax.plot(lstm_result, label='LSTM', color='steelblue', linewidth=0.8, alpha=0.7)
ax.plot(xgb_pred_orig, label='XGBoost', color='darkorange', linewidth=0.8, alpha=0.7)
ax.plot(ensemble_simple, label='Simple average', color='seagreen', linewidth=1.0, alpha=0.9)
ax.plot(ensemble_inverse_rmse, label='Inverse RMSE weighting', color='mediumpurple', linewidth=1.0, alpha=0.9)
ax.plot(ensemble_grid_search, label='Grid-search weight', color='crimson', linewidth=1.0, alpha=0.9)
ax.plot(ensemble_median, label='Median blend', color='gray', linewidth=1.0, alpha=0.9, linestyle='--')
ax.plot(xgb_test_pred_orig, label='XGBoost only', color='navy', linewidth=1.0, alpha=0.85, linestyle=':')
ax.set_title('Precipitation Forecast - Test Set')
ax.set_xlabel('Time step')
ax.set_ylabel('Precipitation (inches)')
ax.legend(ncol=2)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
# ── RMSE bar chart ─────────────────────────────────────────────────────────
models = [
    'LSTM\n(baseline)',
    'XGBoost\n(standalone)',
    'Simple\naverage',
    'Inverse RMSE\nweighting',
    'Grid-search\nweight',
    'Median\nblend',
    'XGBoost\nonly',
]
rmses = [
    score_lstm,
    score_xgb,
    score_simple,
    score_inverse_rmse,
    score_grid,
    score_median,
    score_xgb_only,
]
colors = ['#5b9bd5', '#f0a500', '#2e8b57', '#9370db', '#c0392b', '#7f8c8d', '#1f4e79']

fig, ax = plt.subplots(figsize=(13, 5))
bars = ax.bar(models, rmses, color=colors, edgecolor='white', width=0.55)
for bar, val in zip(bars, rmses):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.05,
            f'{val:.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

ax.set_title('RMSE Comparison Across Ensemble Strategies')
ax.set_ylabel('RMSE (inches)')
ax.set_ylim(0, max(rmses) * 1.15)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()
